# Contratación pública en Bogotá · SECOP II

<a href="https://colab.research.google.com/github/Kzamudioq/SECOP/blob/master/notebooks/analisis_secop.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"></a>

Este cuaderno descarga **todos** los contratos de SECOP II firmados por entidades con sede en Bogotá desde 2021, los limpia con controles de calidad explícitos y responde cinco preguntas:

1. ¿Cuándo se contrata? Estacionalidad y efecto de la **Ley de Garantías**.
2. ¿Cómo se contrata? Peso de la **contratación directa** en número y en valor.
3. ¿En qué se gasta? Sectores y entidades.
4. ¿A quién se le contrata? Proveedores, pymes y **concentración** (HHI).
5. ¿Qué contratos merecen una segunda mirada? **Señales** para revisión.

Toda la lógica vive en el paquete `secop` (carpeta `src/`), probado con datos sintéticos. El cuaderno solo lo llama y comenta los resultados.

## 1. Instalación

In [ ]:
%pip install -q "git+https://github.com/Kzamudioq/SECOP.git"

## 2. Parámetros

Cambia el departamento o las fechas aquí. `MAX_REGISTROS = 20_000` sirve para una prueba rápida; con `None` se descarga el universo completo (varios cientos de miles de contratos, unos minutos).

Si tienes un *app token* de datos.gov.co, ponlo en la variable de entorno `SOCRATA_APP_TOKEN`: la descarga es más rápida y no se corta por límite de peticiones.

In [ ]:
DEPARTAMENTO = "Distrito Capital de Bogotá"
DESDE = "2021-01-01"
HASTA = None            # exclusiva, p. ej. "2025-01-01"
MAX_REGISTROS = None    # 20_000 para probar
ALCANCE = "entidades con sede en Bogotá"

## 3. Descarga

El filtro de fecha y departamento se aplica **en el servidor** y la descarga se pagina hasta traer todo. Solo se piden las 33 columnas que el análisis usa: los datos personales que el dataset publica (representantes legales, cuentas bancarias, supervisores) no se descargan.

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display
from secop import analisis as an, descarga
from secop.limpieza import limpiar, utilizables
from secop.graficas import cop, miles

crudo = descarga.descargar(DEPARTAMENTO, DESDE, HASTA, MAX_REGISTROS)
descarga.guardar(crudo, "datos/contratos.parquet")
print(f"{len(crudo):,} registros")

## 4. Limpieza y calidad

Nada se borra en silencio. Cada problema queda marcado (`q_*`) y contado. Los montos ilegibles quedan vacíos, no en cero; los duplicados se resuelven con la versión más reciente del contrato; las categorías se unifican (`terminado` = `Terminado`).

In [ ]:
df, calidad = limpiar(crudo)
u = utilizables(df)
calidad.tabla()

In [ ]:
r = an.resumen(df)
display(Markdown(f"""
- **{miles(r['contratos_utilizables'])}** contratos con fecha y valor válidos, de **{miles(r['entidades'])}** entidades y **{miles(r['proveedores'])}** proveedores
- Valor total **{cop(r['valor_total'])}** · contrato mediano **{cop(r['valor_mediano'])}**
- Contratación directa: **{r['pct_directa_contratos']:.1f} %** de los contratos y **{r['pct_directa_valor']:.1f} %** del valor
"""))

## 5. Generar todas las figuras y tablas

In [ ]:
from secop import informe
out = informe.generar(df, calidad, "salidas", ALCANCE)
FIG = "salidas/figuras/"

## 6. ¿Cuándo se contrata?

Dos efectos se superponen: el **inicio de vigencia** (en enero se firman los contratos de prestación de servicios del año) y la **Ley de Garantías** (Ley 996 de 2005), que restringe la contratación directa en los cuatro meses previos a las elecciones. La franja naranja marca esas ventanas.

In [ ]:
display(Image(FIG + "01_contratos_por_mes.png"))
display(Image(FIG + "02_estacionalidad.png"))

Para separar los dos efectos, la tabla compara la contratación directa de los 30 días previos a cada veda con **los mismos 30 días de otros años**. Una razón cercana a 1 indica que no hubo adelanto; una razón alta, que las entidades firmaron antes de que empezara la restricción.

In [ ]:
an.efecto_ley_garantias(df)

## 7. ¿Cómo se contrata?

In [ ]:
display(Image(FIG + "03_modalidades.png"))
an.por_modalidad(df).head(8)

La contratación directa suele dominar en **número** (muchos contratos de prestación de servicios con personas naturales) y pesar mucho menos en **valor**. Las licitaciones son pocas pero concentran la plata.

## 8. ¿En qué se gasta?

In [ ]:
display(Image(FIG + "04_sectores.png"))
display(Image(FIG + "06_distribucion_valor.png"))
an.duraciones(df).head(8)

## 9. ¿A quién se le contrata?

Los proveedores se agrupan por **documento**, no por nombre: así "EMPRESA S.A.S." y "EMPRESA SAS" cuentan como uno. Las personas naturales se muestran enmascaradas.

La concentración de cada entidad se mide con el **índice Herfindahl-Hirschman** sobre el valor adjudicado a personas jurídicas: por debajo de 1.500 es baja, entre 1.500 y 2.500 moderada y por encima de 2.500 alta.

In [ ]:
display(Image(FIG + "07_top_proveedores.png"))
display(Image(FIG + "05_concentracion.png"))
an.concentracion_entidades(df).head(10)

In [ ]:
an.pymes(df).head(8)

## 10. Señales para revisión

Cuatro señales, contrato por contrato:

| Señal | Regla |
|---|---|
| `valor_atipico` | el log del valor se aleja más de 3,5 desviaciones robustas (MAD) de la mediana de su tipo y modalidad |
| `pagado_supera_valor` | se pagó más de lo pactado |
| `adicion_mayor_50` | los días adicionados superan la mitad del plazo original |
| `inicio_antes_firma` | la ejecución empezó antes de la firma |

**Una señal no es una irregularidad.** Muchas tienen explicación (errores de digitación, contratos marco, adiciones justificadas). Sirven para priorizar qué mirar en el expediente de SECOP.

In [ ]:
alertas = an.alertas(df)
display(an.resumen_alertas(alertas, len(u)))
alertas.head(15)

## 11. Resultados

Todo queda en `salidas/`: `informe.md`, las figuras en `figuras/` y las tablas en `tablas/secop_resultados.xlsx` (una hoja por análisis) y en CSV.

In [ ]:
import shutil
shutil.make_archive("secop_salidas", "zip", "salidas")
try:
    from google.colab import files
    files.download("secop_salidas.zip")
except ImportError:
    print("Resultados en ./salidas y ./secop_salidas.zip")